In [ ]:
import os
import sys
import joblib
import logging
import numpy as np
import pandas as pd
import rasterio
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)

logging.info("Logging is working ✅")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/PhytoLB"

GBIF_PATH = f"{BASE_DIR}/dwca-tree_lebanon-v1.0/occurrence.txt"
CLIMATE_DIR = f"{BASE_DIR}/wc2.1_10m_bio"
ELEVATION_PATH = f"{BASE_DIR}/elevation/elevation.tif"

SAVE_DIR = f"{BASE_DIR}/model_enhanced_v3"
os.makedirs(SAVE_DIR, exist_ok=True)

logging.info(f"Saving outputs to: {SAVE_DIR}")

In [ ]:
def load_gbif_txt(path):
    logging.info("Loading GBIF occurrence.txt file...")

    df = pd.read_csv(path, sep="\t", low_memory=False)

    if "species" not in df.columns:
        df["species"] = df["scientificName"]

    df = df[["species", "decimalLatitude", "decimalLongitude"]]
    df = df.dropna()

    df.columns = ["species", "lat", "lon"]

    # Keep Lebanon approximate bounds
    df = df[
        (df["lat"] >= 33.0) & (df["lat"] <= 34.8) &
        (df["lon"] >= 35.0) & (df["lon"] <= 36.8)
    ]

    df = df.drop_duplicates()

    logging.info(f"GBIF loaded and cleaned: {len(df)} records")
    logging.info(f"Unique species: {df['species'].nunique()}")

    return df

In [ ]:
class EnvironmentalExtractor:
    def __init__(self, climate_folder, elevation_path=None):
        self.climate_rasters = []

        logging.info("Opening WorldClim BIO rasters...")

        for i in range(1, 20):
            path = os.path.join(
                climate_folder,
                f"wc2.1_10m_bio_{i}.tif"
            )

            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing climate file: {path}")

            self.climate_rasters.append(rasterio.open(path))

        logging.info(f"Loaded {len(self.climate_rasters)} climate rasters")

        self.elevation_raster = None

        if elevation_path is not None:
            if not os.path.exists(elevation_path):
                raise FileNotFoundError(f"Missing elevation file: {elevation_path}")

            self.elevation_raster = rasterio.open(elevation_path)
            logging.info("Elevation raster loaded ✅")

    def extract_point(self, lat, lon):
        values = []

        # Extract BIO1-BIO19
        for raster in self.climate_rasters:
            try:
                val = next(raster.sample([(lon, lat)]))[0]

                if raster.nodata is not None and val == raster.nodata:
                    val = np.nan

            except Exception:
                val = np.nan

            values.append(val)

        # Extract elevation
        if self.elevation_raster is not None:
            try:
                elev = next(self.elevation_raster.sample([(lon, lat)]))[0]

                if (
                    self.elevation_raster.nodata is not None
                    and elev == self.elevation_raster.nodata
                ):
                    elev = np.nan

            except Exception:
                elev = np.nan

            values.append(elev)

        return values

    def extract_dataframe(self, df):
        features = []
        total = len(df)

        logging.info(f"Extracting environmental features for {total} points...")

        for i, row in enumerate(df.itertuples(index=False)):
            vals = self.extract_point(row.lat, row.lon)
            features.append(vals)

            if i % 1000 == 0:
                logging.info(f"Processed {i}/{total} points")

        feature_names = [f"BIO{i}" for i in range(1, 20)]

        if self.elevation_raster is not None:
            feature_names.append("elevation")

        return np.array(features), feature_names

In [ ]:
def generate_pseudo_absence(df, n_samples, random_state=42):
    np.random.seed(random_state)

    lat_min, lat_max = df["lat"].min(), df["lat"].max()
    lon_min, lon_max = df["lon"].min(), df["lon"].max()

    pseudo = pd.DataFrame({
        "species": df["species"].sample(
            n_samples,
            replace=True,
            random_state=random_state
        ).values,
        "lat": np.random.uniform(lat_min, lat_max, n_samples),
        "lon": np.random.uniform(lon_min, lon_max, n_samples),
        "label": 0
    })

    return pseudo

In [ ]:
def build_dataset(df, extractor):
    logging.info("Building presence dataset...")

    X_presence, feature_names = extractor.extract_dataframe(df)

    df_presence = pd.DataFrame(
        X_presence,
        columns=feature_names
    )

    df_presence["species"] = df["species"].values
    df_presence["label"] = 1

    logging.info("Generating pseudo-absence points...")

    pseudo_df = generate_pseudo_absence(
        df,
        n_samples=len(df),
        random_state=42
    )

    logging.info("Building pseudo-absence dataset...")

    X_absence, _ = extractor.extract_dataframe(pseudo_df)

    df_absence = pd.DataFrame(
        X_absence,
        columns=feature_names
    )

    df_absence["species"] = pseudo_df["species"].values
    df_absence["label"] = 0

    full_df = pd.concat(
        [df_presence, df_absence],
        ignore_index=True
    )

    return full_df

In [ ]:
def get_model():
    return lgb.LGBMClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=8,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=42,
        force_row_wise=True,
        verbose=-1
    )

In [ ]:
gbif = load_gbif_txt(GBIF_PATH)

gbif.to_csv(f"{SAVE_DIR}/clean_gbif.csv", index=False)
logging.info("Clean GBIF saved ✅")


extractor = EnvironmentalExtractor(
    climate_folder=CLIMATE_DIR,
    elevation_path=ELEVATION_PATH
)

dataset = build_dataset(gbif, extractor)

logging.info(f"Dataset built: {dataset.shape}")
logging.info(f"Columns: {dataset.columns.tolist()}")
logging.info(f"Class distribution:\n{dataset['label'].value_counts()}")

missing = dataset.isna().sum().sum()
logging.info(f"Total missing values: {missing}")

if "elevation" in dataset.columns:
    logging.info(f"Elevation summary:\n{dataset['elevation'].describe()}")

dataset.to_csv(f"{SAVE_DIR}/final_phyto_dataset.csv", index=False)
dataset.to_parquet(f"{SAVE_DIR}/final_phyto_dataset.parquet", index=False)

logging.info("Final dataset saved ✅")


X = dataset.drop(columns=["label"])
y = dataset["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logging.info(f"Train size: {X_train.shape}")
logging.info(f"Test size: {X_test.shape}")

X_train.to_csv(f"{SAVE_DIR}/X_train_raw.csv", index=False)
X_test.to_csv(f"{SAVE_DIR}/X_test_raw.csv", index=False)
y_train.to_csv(f"{SAVE_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{SAVE_DIR}/y_test.csv", index=False)


le = LabelEncoder()

X_train = X_train.copy()
X_test = X_test.copy()

X_train["species_id"] = le.fit_transform(X_train["species"])
X_test["species_id"] = le.transform(X_test["species"])

X_train = X_train.drop(columns=["species"])
X_test = X_test.drop(columns=["species"])

feature_names = X_train.columns.tolist()

logging.info(f"Encoded species count: {len(le.classes_)}")
logging.info(f"Final feature names: {feature_names}")


imputer = SimpleImputer(strategy="mean")

X_train_processed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_names,
    index=X_test.index
)

logging.info(f"NaNs train after imputation: {X_train_processed.isna().sum().sum()}")
logging.info(f"NaNs test after imputation: {X_test_processed.isna().sum().sum()}")

X_train_processed.to_csv(f"{SAVE_DIR}/X_train_processed.csv", index=False)
X_test_processed.to_csv(f"{SAVE_DIR}/X_test_processed.csv", index=False)


model = get_model()

logging.info("Training LightGBM model...")
model.fit(X_train_processed, y_train)

logging.info("Model training completed ✅")


joblib.dump(model, f"{SAVE_DIR}/phyto_lightgbm_model.pkl")
joblib.dump(le, f"{SAVE_DIR}/species_encoder.pkl")
joblib.dump(imputer, f"{SAVE_DIR}/imputer.pkl")
joblib.dump(feature_names, f"{SAVE_DIR}/feature_names.pkl")

logging.info("Model artifacts saved ✅")


y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

report = classification_report(y_test, y_pred)

logging.info(f"Accuracy: {acc:.4f}")
logging.info(f"AUC: {auc:.4f}")

print("\nClassification Report:\n")
print(report)

with open(f"{SAVE_DIR}/evaluation_report.txt", "w") as f:
    f.write(f"Accuracy: {acc:.4f}\n")
    f.write(f"AUC: {auc:.4f}\n\n")
    f.write(report)

logging.info("Evaluation report saved ✅")


def suitability_class(prob):
    if prob >= 0.75:
        return "Highly suitable"
    elif prob >= 0.50:
        return "Suitable"
    elif prob >= 0.30:
        return "Moderately suitable"
    else:
        return "Low / not suitable"


def predict_species_suitability(city_name, lat, lon, species_name):
    if species_name not in le.classes_:
        raise ValueError(
            f"Species '{species_name}' not found. "
            f"Available example: {le.classes_[:5]}"
        )

    env_values = extractor.extract_point(lat, lon)

    sample_df = pd.DataFrame(
        [env_values],
        columns=[f"BIO{i}" for i in range(1, 20)] + (
            ["elevation"] if "elevation" in feature_names else []
        )
    )

    sample_df["species_id"] = le.transform([species_name])[0]

    sample_df = sample_df[feature_names]

    sample_processed = pd.DataFrame(
        imputer.transform(sample_df),
        columns=feature_names
    )

    prob = model.predict_proba(sample_processed)[0][1]
    label = int(prob >= 0.5)

    print("====================================")
    print(f"City: {city_name}")
    print(f"Species: {species_name}")
    print(f"Suitability Probability: {prob:.4f}")
    print(f"Suitability Class: {suitability_class(prob)}")
    print(f"Binary Prediction: {'Suitable' if label == 1 else 'Not suitable'}")
    print("====================================")

    return prob, label


predict_species_suitability(
    city_name="Cedars of God",
    lat=34.2436,
    lon=36.0486,
    species_name="Cedrus libani"
)

logging.info("Pipeline completed successfully 🚀")

2026-05-10 05:49:56,502 - INFO - Logging is working ✅
2026-05-10 05:49:56,508 - INFO - Saving outputs to: /content/drive/MyDrive/PhytoLB/model_enhanced_v3
2026-05-10 05:49:56,510 - INFO - Loading GBIF occurrence.txt file...
2026-05-10 05:49:57,011 - INFO - GBIF loaded and cleaned: 13748 records
2026-05-10 05:49:57,014 - INFO - Unique species: 27
2026-05-10 05:49:57,071 - INFO - Clean GBIF saved ✅
2026-05-10 05:49:57,071 - INFO - Opening WorldClim BIO rasters...
2026-05-10 05:50:02,333 - INFO - Loaded 19 climate rasters
2026-05-10 05:50:03,203 - INFO - Elevation raster loaded ✅
2026-05-10 05:50:03,205 - INFO - Building presence dataset...
2026-05-10 05:50:03,206 - INFO - Extracting environmental features for 13748 points...
2026-05-10 05:50:04,158 - INFO - Processed 0/13748 points
2026-05-10 05:50:08,801 - INFO - Processed 1000/13748 points
2026-05-10 05:50:11,499 - INFO - Processed 2000/13748 points
2026-05-10 05:50:14,213 - INFO - Processed 3000/13748 points
2026-05-10 05:50:17,143 - 